# Redrob Ranker Sandbox Demo

This notebook verifies that the repository can install dependencies, run the ranking step, patch the audited reasoning rows, validate the CSV, and preview the output.

Set `REPO_URL` before running if the repository is hosted on GitHub. If the repository is already uploaded into the Colab filesystem, leave `REPO_URL` as `TODO` and set `PROJECT_DIR` manually.

In [ ]:
from pathlib import Path
import os

REPO_URL = "TODO"  # Example: "https://github.com/<user>/<repo>.git"
PROJECT_DIR = Path("/content/redrob-ranker")

if REPO_URL != "TODO":
    if PROJECT_DIR.exists():
        print(f"Using existing directory: {PROJECT_DIR}")
    else:
        !git clone "$REPO_URL" "$PROJECT_DIR"
else:
    print("REPO_URL is TODO. Upload or clone the repository, then set PROJECT_DIR to that folder.")

print("PROJECT_DIR =", PROJECT_DIR)

In [ ]:
%cd $PROJECT_DIR
!python --version
!python -m pip install -q -r requirements.txt

In [ ]:
from pathlib import Path

required = [
    "rank.py",
    "patch_reasoning.py",
    "validate_submission.py",
    "requirements.txt",
    "candidates.jsonl",
    "job_description.docx",
    "artifacts/candidate_embeddings.npy",
    "artifacts/candidate_ids.json",
    "artifacts/candidates_parsed.pkl",
    "artifacts/bm25s_index",
    "models/bge_small/config.json",
    "models/bge_reranker/config.json",
]

missing = [p for p in required if not Path(p).exists()]
if missing:
    raise FileNotFoundError("Missing required files: " + ", ".join(missing))
print("Required files found.")

In [ ]:
!python rank.py --candidates ./candidates.jsonl --job-description ./job_description.docx --cross-encoder-limit 30 --out ./submission.csv
!python patch_reasoning.py
!python validate_submission.py --submission ./submission.csv

In [ ]:
import csv

with open("submission.csv", newline="", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

print("rows:", len(rows))
for row in rows[:5]:
    print(row["rank"], row["candidate_id"], row["score"], row["reasoning"][:120])